<a href="https://colab.research.google.com/github/0xdaxl/soc-ml-recommendation-engine/blob/main/notebooks/SOC_ML_ngrok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-generativeai flask pyngrok

In [2]:
from google import genai
from google.colab import userdata
import json
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok

# ============================================================
# CONFIGURATION
# ============================================================
API_KEY = userdata.get('GEMINI_API_KEY')
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
client = genai.Client(api_key=API_KEY)

# ============================================================
# COMPLIANCE RULES
# ============================================================
compliance_map = {
    "brute_force": """
HIPAA §164.312(d) — Person/Entity Authentication:
Implement procedures to verify that a person or entity seeking access
to ePHI is who they claim to be. Failed authentication must be monitored.

HIPAA §164.308(a)(5)(ii)(C) — Log-in Monitoring:
Procedures for monitoring log-in attempts and reporting discrepancies.

NIST AC-7 — Unsuccessful Logon Attempts:
Enforce a limit on consecutive invalid logon attempts. Lock the account
after the threshold is exceeded.

NIST IA-5 — Authenticator Management:
Manage system authenticators including passwords, tokens, and certificates.
    """,
    "malware": """
HIPAA §164.308(a)(1)(ii)(A) — Risk Analysis:
Conduct an accurate and thorough assessment of potential risks to ePHI.

HIPAA §164.308(a)(5)(ii)(B) — Protection from Malicious Software:
Procedures for guarding against, detecting, and reporting malicious software.

NIST SI-3 — Malicious Code Protection:
Implement malicious code protection at system entry/exit points.
Take action when malicious code is detected including alerting administrators.

NIST IR-4 — Incident Handling:
Implement an incident handling capability including preparation,
detection, analysis, containment, eradication, and recovery.
    """,
    "privilege_escalation": """
HIPAA §164.312(a)(1) — Access Control — Unique User Identification:
Assign a unique name/number for identifying and tracking user identity.
Clinician accounts must only have access required for their role.

HIPAA §164.308(a)(3) — Workforce Access Management:
Implement policies to authorize and supervise workforce access to ePHI.

NIST AC-6 — Least Privilege:
Employ the principle of least privilege, allowing only authorized access
required for users to accomplish assigned tasks.

NIST AC-2 — Account Management:
Manage system accounts including establishing, activating, and monitoring.
    """,
    "generic": """
HIPAA §164.308(a)(1) — Security Management Process:
Implement policies and procedures to prevent, detect, contain, and
correct security violations.

NIST IR-4 — Incident Handling:
Implement an incident handling capability for security incidents.

NIST SI-5 — Security Alerts and Advisories:
Receive information system security alerts and take appropriate actions.
    """
}

# ============================================================
# ALERT TYPE DETECTION
# ============================================================
def detect_alert_type(alert):
    groups = alert.get("rule", {}).get("groups", [])
    if "brute_force" in groups or "authentication_failures" in groups:
        return "brute_force"
    elif "malware" in groups or "syscheck" in groups:
        return "malware"
    elif "privilege_escalation" in groups or "sudo" in groups:
        return "privilege_escalation"
    else:
        return "generic"

# ============================================================
# PROMPT BUILDER
# ============================================================
def build_prompt(alert, alert_type):
    compliance = compliance_map[alert_type]
    alert_str = json.dumps(alert, indent=2)
    prompt = f"""You are an expert SOC analyst specializing in healthcare cybersecurity.
You have deep knowledge of HIPAA Security Rule and NIST SP 800-53 controls.
You are analyzing alerts from a Wazuh SIEM deployed in a healthcare environment
that handles Electronic Health Records (EHR) and Protected Health Information (PHI).
Your job is to give clear, actionable recommendations to SOC analysts.

RELEVANT COMPLIANCE RULES FOR THIS ALERT:
{compliance}

WAZUH ALERT TO ANALYZE:
{alert_str}

Provide your analysis in EXACTLY this format:

**WHAT HAPPENED:** (2-3 sentences in plain language)
**THREAT LEVEL:** (CRITICAL / HIGH / MEDIUM / LOW + one sentence justification)
**COMPLIANCE VIOLATION:** (Specific HIPAA section and NIST control violated)
**IMMEDIATE ACTIONS:** (Numbered list — next 15 minutes)
**INVESTIGATION STEPS:** (Numbered list — what to check next)
**CASE NOTES FOR THEHIVE:** (Short paragraph ready to paste)
"""
    return prompt

# ============================================================
# RECOMMENDATION ENGINE
# ============================================================
def get_recommendation(alert):
    alert_type = detect_alert_type(alert)
    prompt = build_prompt(alert, alert_type)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

# ============================================================
# FLASK WEB SERVER
# ============================================================
app = Flask(__name__)

@app.route('/recommend', methods=['POST'])
def recommend():
    alert = request.json
    if not alert:
        return jsonify({"error": "No alert data received"}), 400
    print(f"\n[ALERT RECEIVED] Rule: {alert.get('rule', {}).get('description', 'unknown')}")
    recommendation = get_recommendation(alert)
    print(f"[RECOMMENDATION SENT] {recommendation[:100]}...")
    return jsonify({
        "status": "success",
        "alert_type": detect_alert_type(alert),
        "recommendation": recommendation
    })

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ML engine running"})

# ============================================================
# NGROK TUNNEL
# ============================================================
threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False)
).start()

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(5000)
public_url = tunnel.public_url

print("\n" + "="*60)
print("ML ENGINE IS RUNNING")
print("="*60)
print(f"Public URL:     {public_url}")
print(f"n8n endpoint:   {public_url}/recommend")
print(f"Health check:   {public_url}/health")
print("="*60)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



ML ENGINE IS RUNNING
Public URL:     https://riteless-andree-techily.ngrok-free.dev
n8n endpoint:   https://riteless-andree-techily.ngrok-free.dev/recommend
Health check:   https://riteless-andree-techily.ngrok-free.dev/health
